# JS08 — Lab 2: HDBSCAN vs DBSCAN

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan DBSCAN dan HDBSCAN pada data sintetis; menunjukkan bahwa DBSCAN sensitif terhadap skala data dan nilai `eps`, sedangkan HDBSCAN (*hierarchical density-based*) lebih adaptif terhadap cluster dengan kepadatan berbeda.

**Prasyarat:** tidak perlu dataset eksternal — data dibangkitkan dengan `make_blobs`. Library `hdbscan` dipasang pada cell pertama.


# Lab 2 — HDBSCAN

Membandingkan **DBSCAN** dan **HDBSCAN**. DBSCAN sensitif terhadap `eps` dan skala data; HDBSCAN membangun hirarki kepadatan sehingga lebih adaptif terhadap cluster dengan kepadatan berbeda dan tidak menuntut nilai `eps`.

## Langkah 0 — Persiapan Lingkungan

In [ ]:
# Instalasi library hdbscan (tidak tersedia default di sklearn)
# Dijalankan sekali saja; aman diabaikan bila hdbscan sudah terpasang.
try:
    import hdbscan  # noqa: F401
    print("hdbscan sudah terpasang")
except ImportError:
    %pip install -q hdbscan

In [ ]:
# Import modul
import matplotlib.pyplot as plt
import numpy as np

from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
import hdbscan

## Langkah 1 — Membuat Fungsi Visualisasi

Fungsi `plot()` memberi warna berbeda per cluster, menandai noise (`-1`) dengan tanda **x** hitam, dan mengatur ukuran marker sesuai nilai probabilitas keanggotaan.

In [ ]:
def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    labels = labels if labels is not None else np.ones(X.shape[0])
    probabilities = probabilities if probabilities is not None else np.ones(X.shape[0])
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    proba_map = {idx: probabilities[idx] for idx in range(len(labels))}

    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = [0, 0, 0, 1]  # warna hitam untuk noise
        class_index = (labels == k).nonzero()[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0],
                X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=4 if k == -1 else 1 + 5 * proba_map[ci],
            )

    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" | {parameters_str}"
    ax.set_title(title)
    plt.tight_layout()

## Langkah 2 — Membuat Data Sintetis

Dataset terdiri dari 3 cluster Gaussian dengan *spread* berbeda.

In [ ]:
centers = [[1, 1], [-1, -1], [1.5, -1.5]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.4, 0.1, 0.75], random_state=0
)

plot(X, labels=labels_true, ground_truth=True)

## Langkah 3 — Uji *Scale Invariance* pada DBSCAN

`eps=0.3` dijalankan pada data dengan skala 1×, 0.5×, dan 3×.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
dbs = DBSCAN(eps=0.3)
for idx, scale in enumerate([1, 0.5, 3]):
    dbs.fit(X * scale)
    plot(X * scale, dbs.labels_, parameters={"scale": scale, "eps": 0.3}, ax=axes[idx])

**Pengamatan:** hasil DBSCAN berubah drastis hanya karena skala data — pada skala 0.5× semua titik menjadi noise, pada skala 3× cluster menyatu. `eps` harus disesuaikan dengan skala.

### Perbaikan: `eps` disesuaikan dengan skala

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(12, 5))
dbs = DBSCAN(eps=0.9).fit(3 * X)
plot(3 * X, dbs.labels_, parameters={"scale": 3, "eps": 0.9}, ax=axis)

## Langkah 4 — HDBSCAN pada Skala Berbeda

HDBSCAN dijalankan pada data yang sama dengan skala 1×, 0.5×, dan 3× tanpa menyetel parameter jarak.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
hdb = hdbscan.HDBSCAN()
for idx, scale in enumerate([1, 0.5, 3]):
    hdb.fit(X * scale)
    plot(X * scale, hdb.labels_, hdb.probabilities_, ax=axes[idx], parameters={"scale": scale})

**Pengamatan:** struktur cluster HDBSCAN tetap konsisten pada ketiga skala (*scale invariant*) — inilah keunggulan utamanya dibanding DBSCAN.

## Langkah 5 — *Multiscale* Clustering

Dataset baru dengan dua cluster rapat (kepadatan tinggi) dan dua cluster renggang (kepadatan rendah).

In [ ]:
centers = [[-0.85, -0.85], [-0.85, 0.85], [3, 3], [3, -3]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.2, 0.35, 1.35, 1.35], random_state=0
)

plot(X, labels=labels_true, ground_truth=True)

### Perbandingan DBSCAN dengan `eps` berbeda

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

params = {"eps": 0.7}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[0])

params = {"eps": 0.3}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[1])

**Pengamatan:** satu nilai `eps` tidak bisa benar untuk keduanya. `eps` besar menggabungkan cluster renggang menjadi satu, `eps` kecil membuat cluster renggang dianggap noise.

### HDBSCAN pada dataset *multiscale*

In [ ]:
hdb = hdbscan.HDBSCAN().fit(X)
plot(X, hdb.labels_, hdb.probabilities_)

In [ ]:
# Jumlah data per cluster (label -1 = noise)
import pandas as pd

pd.Series(hdb.labels_).value_counts().sort_index().rename_axis('label').to_frame('jumlah')

---
## Kesimpulan Lab 2

**Pengaruh skala terhadap DBSCAN:** TBD

**Mengapa HDBSCAN lebih unggul pada dataset multiscale:** TBD

*(Sertakan bukti: jumlah cluster/noise yang berubah pada `eps` 0.7 vs 0.3, dan hasil stabil HDBSCAN.)*